# Fine-tuning SegFormer for underwater semantic segmentation

This notebook is the semantic segmentation companion to the underwater image classification exercise. It follows the Hugging Face semantic segmentation tutorial closely, but replaces SceneParse150 with the underwater [SUIM dataset](https://irvlab.cs.umn.edu/resources/suim-dataset).

Semantic segmentation assigns a class label to every pixel. In SUIM, the classes describe underwater scene elements such as water/background, divers, plants, wrecks, robots/instruments, reefs/invertebrates, fish, and sea floor/rocks.

The model is [`nvidia/mit-b0`](https://huggingface.co/nvidia/mit-b0), the small SegFormer encoder used in the Hugging Face tutorial. We fine-tune it through `AutoModelForSemanticSegmentation` and evaluate with mean Intersection over Union (mean IoU).


## Setup

Run this notebook on a GPU machine. Do not train it on the course laptop if no GPU is available.

Install dependencies from the accompanying `requirements.txt`:

```bash
micromamba create -n underwater-ml python=3.11 pip
micromamba activate underwater-ml
pip install -r 03a-underwater/requirements.txt
```


In [ ]:
from inspect import signature
from pathlib import Path

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import Dataset, DatasetDict, Image
from PIL import Image as PILImage
from torch import nn
from torchvision.transforms import ColorJitter
from transformers import (
    AutoImageProcessor,
    AutoModelForSemanticSegmentation,
    Trainer,
    TrainingArguments,
)

seed = 42
rng = np.random.default_rng(seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
device


## Download SUIM

Download SUIM from the official dataset page and extract it locally:

- Dataset page: https://irvlab.cs.umn.edu/resources/suim-dataset
- Paper: https://arxiv.org/abs/2004.01241

Expected folder layout after extraction:

```text
SUIM/
  train_val/
    images/*.jpg
    masks/*.bmp or *.png
  TEST/
    images/*.jpg
    masks/*.bmp or *.png
```

If your extracted archive uses slightly different folder names, adjust `suim_root`, `train_split_name`, and `test_split_name` in the next cell.


In [ ]:
suim_root = Path("SUIM")
train_split_name = "train_val"
test_split_name = "TEST"

train_image_dir = suim_root / train_split_name / "images"
train_mask_dir = suim_root / train_split_name / "masks"
test_image_dir = suim_root / test_split_name / "images"
test_mask_dir = suim_root / test_split_name / "masks"

for path in [train_image_dir, train_mask_dir, test_image_dir, test_mask_dir]:
    print(path, "exists=" + str(path.exists()))


## Labels

SUIM uses eight semantic classes. Class `0` is a real class (`background_waterbody`), so unlike ADE20K in the Hugging Face tutorial, we set `do_reduce_labels=False` later.


In [ ]:
id2label = {
    0: "background_waterbody",
    1: "human_diver",
    2: "plants_seagrass",
    3: "wreck_ruin",
    4: "robot_instrument",
    5: "reef_invertebrate",
    6: "fish_vertebrate",
    7: "seafloor_rock",
}
label2id = {label: idx for idx, label in id2label.items()}
num_labels = len(id2label)
num_labels, id2label


## Load SUIM as a Hugging Face `DatasetDict`

This mirrors the custom-dataset section of the Hugging Face semantic segmentation tutorial: build datasets with two image columns, `image` and `annotation`, then cast both columns to `datasets.Image`.


In [ ]:
def list_files(directory, suffixes=(".jpg", ".jpeg", ".png", ".bmp")):
    return sorted(
        path for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in suffixes
    )


def pair_images_and_masks(image_dir, mask_dir):
    image_paths = list_files(image_dir, suffixes=(".jpg", ".jpeg", ".png"))
    mask_paths = list_files(mask_dir, suffixes=(".png", ".bmp", ".jpg", ".jpeg"))
    masks_by_stem = {path.stem: path for path in mask_paths}

    paired_images = []
    paired_masks = []
    missing = []
    for image_path in image_paths:
        mask_path = masks_by_stem.get(image_path.stem)
        if mask_path is None:
            missing.append(image_path.name)
            continue
        paired_images.append(str(image_path))
        paired_masks.append(str(mask_path))

    if missing:
        raise ValueError(f"Missing masks for {len(missing)} images, for example: {missing[:5]}")
    if not paired_images:
        raise ValueError(f"No image/mask pairs found in {image_dir} and {mask_dir}")
    return paired_images, paired_masks


def create_dataset(image_dir, mask_dir):
    image_paths, mask_paths = pair_images_and_masks(image_dir, mask_dir)
    dataset = Dataset.from_dict({"image": image_paths, "annotation": mask_paths})
    dataset = dataset.cast_column("image", Image())
    dataset = dataset.cast_column("annotation", Image())
    return dataset


suim_dataset = DatasetDict({
    "train": create_dataset(train_image_dir, train_mask_dir),
    "test": create_dataset(test_image_dir, test_mask_dir),
})

# Use small subsets first for a quick classroom dry-run. Set these to None for the full dataset.
max_train_samples = 100
max_test_samples = 40

if max_train_samples is not None:
    suim_dataset["train"] = suim_dataset["train"].shuffle(seed=seed).select(range(min(max_train_samples, len(suim_dataset["train"]))))
if max_test_samples is not None:
    suim_dataset["test"] = suim_dataset["test"].shuffle(seed=seed).select(range(min(max_test_samples, len(suim_dataset["test"]))))

train_ds = suim_dataset["train"]
test_ds = suim_dataset["test"]

suim_dataset


## Convert SUIM masks to class ids

The model expects one integer class id per pixel. SUIM annotations are distributed as color masks. The helper below converts the binary RGB code used by SUIM into ids `0..7`. It supports masks whose foreground channels are stored as either `1` or `255`.


In [ ]:
suim_color_to_id = {
    (0, 0, 0): 0,
    (0, 0, 1): 1,
    (0, 1, 0): 2,
    (0, 1, 1): 3,
    (1, 0, 0): 4,
    (1, 0, 1): 5,
    (1, 1, 0): 6,
    (1, 1, 1): 7,
    (0, 0, 255): 1,
    (0, 255, 0): 2,
    (0, 255, 255): 3,
    (255, 0, 0): 4,
    (255, 0, 255): 5,
    (255, 255, 0): 6,
    (255, 255, 255): 7,
}

suim_palette = np.array([
    [0, 0, 0],
    [0, 0, 255],
    [0, 255, 0],
    [0, 255, 255],
    [255, 0, 0],
    [255, 0, 255],
    [255, 255, 0],
    [255, 255, 255],
], dtype=np.uint8)


def rgb_mask_to_class_ids(mask):
    mask = np.array(mask.convert("RGB"))
    class_mask = np.zeros(mask.shape[:2], dtype=np.uint8)
    for color, class_id in suim_color_to_id.items():
        class_mask[np.all(mask == color, axis=-1)] = class_id
    return PILImage.fromarray(class_mask, mode="L")


def class_ids_to_rgb(mask):
    mask = np.array(mask, dtype=np.uint8)
    return suim_palette[mask]


In [ ]:
example = train_ds[0]
image = example["image"].convert("RGB")
annotation = rgb_mask_to_class_ids(example["annotation"])

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(image)
axes[0].set_title("image")
axes[1].imshow(annotation, vmin=0, vmax=num_labels - 1)
axes[1].set_title("class ids")
axes[2].imshow(np.array(image) * 0.55 + class_ids_to_rgb(annotation) * 0.45).astype(np.uint8)
axes[2].set_title("overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()


## Task

Fine-tune SegFormer on SUIM and evaluate semantic segmentation quality.

1. Load the SegFormer image processor from `nvidia/mit-b0`.
2. Apply `ColorJitter` augmentation to training images.
3. Use `set_transform` so preprocessing happens lazily.
4. Load `AutoModelForSemanticSegmentation` with the SUIM label dictionaries.
5. Train with `Trainer`.
6. Evaluate with mean IoU and mean accuracy.
7. Run inference on one test image and visualize the predicted segmentation mask.

The complete solution is below.


## Solution: preprocess

This follows the Hugging Face tutorial: load the image processor, define separate train and validation transforms, and attach them to the datasets with lazy transforms. Here we use `with_transform` so the raw SUIM dataset remains available for visualization and inference later.


In [ ]:
checkpoint = "nvidia/mit-b0"
image_processor = AutoImageProcessor.from_pretrained(checkpoint, do_reduce_labels=False)

jitter = ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.1)


def train_transforms(example_batch):
    images = [jitter(image.convert("RGB")) for image in example_batch["image"]]
    labels = [rgb_mask_to_class_ids(mask) for mask in example_batch["annotation"]]
    inputs = image_processor(images=images, segmentation_maps=labels)
    return inputs


def val_transforms(example_batch):
    images = [image.convert("RGB") for image in example_batch["image"]]
    labels = [rgb_mask_to_class_ids(mask) for mask in example_batch["annotation"]]
    inputs = image_processor(images=images, segmentation_maps=labels)
    return inputs


train_ds = train_ds.with_transform(train_transforms)
test_ds = test_ds.with_transform(val_transforms)


## Solution: evaluate

Mean Intersection over Union is the standard semantic segmentation metric. The model outputs lower-resolution logits, so we upsample logits to the label size before computing the metric, exactly as in the Hugging Face tutorial.


In [ ]:
metric = evaluate.load("mean_iou")


def compute_metrics(eval_pred):
    with torch.no_grad():
        logits, labels = eval_pred
        logits_tensor = torch.from_numpy(logits)
        logits_tensor = nn.functional.interpolate(
            logits_tensor,
            size=labels.shape[-2:],
            mode="bilinear",
            align_corners=False,
        ).argmax(dim=1)

        pred_labels = logits_tensor.detach().cpu().numpy()
        metrics = metric.compute(
            predictions=pred_labels,
            references=labels,
            num_labels=num_labels,
            ignore_index=255,
            reduce_labels=False,
        )

        # Trainer logging expects scalar-like values. Keep class-wise arrays readable.
        for key, value in metrics.items():
            if isinstance(value, np.ndarray):
                metrics[key] = value.tolist()
        return metrics


## Solution: train

The important Hugging Face `Trainer` detail for image segmentation is `remove_unused_columns=False`. Without it, the `image` and `annotation` columns can be dropped before the transform creates `pixel_values` and `labels`.


In [ ]:
model = AutoModelForSemanticSegmentation.from_pretrained(
    checkpoint,
    id2label=id2label,
    label2id=label2id,
    num_labels=num_labels,
    ignore_mismatched_sizes=True,
)

eval_argument_name = "eval_strategy" if "eval_strategy" in signature(TrainingArguments.__init__).parameters else "evaluation_strategy"

training_args = TrainingArguments(
    output_dir="segformer-b0-suim",
    learning_rate=6e-5,
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    save_total_limit=3,
    save_strategy="steps",
    save_steps=50,
    eval_steps=50,
    logging_steps=10,
    eval_accumulation_steps=5,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    push_to_hub=False,
    **{eval_argument_name: "steps"},
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

# This is the expensive cell. Run it on a GPU.
trainer.train()


In [ ]:
trainer.evaluate()


## Solution: inference

The inference code also mirrors the Hugging Face tutorial: process the image, run the model, upsample logits to the original image size, take the `argmax`, and overlay a color-coded segmentation map.


In [ ]:
raw_test_example = suim_dataset["test"][0]
image = raw_test_example["image"].convert("RGB")
true_mask = rgb_mask_to_class_ids(raw_test_example["annotation"])

encoding = image_processor(image, return_tensors="pt")
pixel_values = encoding.pixel_values.to(model.device)

with torch.no_grad():
    outputs = model(pixel_values=pixel_values)
    logits = outputs.logits.cpu()

upsampled_logits = nn.functional.interpolate(
    logits,
    size=image.size[::-1],
    mode="bilinear",
    align_corners=False,
)
pred_seg = upsampled_logits.argmax(dim=1)[0].numpy().astype(np.uint8)

pred_rgb = class_ids_to_rgb(pred_seg)
true_rgb = class_ids_to_rgb(true_mask)
overlay = (np.array(image) * 0.55 + pred_rgb * 0.45).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(image)
axes[0].set_title("image")
axes[1].imshow(true_rgb)
axes[1].set_title("ground truth")
axes[2].imshow(pred_rgb)
axes[2].set_title("prediction")
axes[3].imshow(overlay)
axes[3].set_title("prediction overlay")
for ax in axes:
    ax.axis("off")
plt.tight_layout()


## Optional: save the fine-tuned model

After a successful GPU run, save the model and processor locally. If you want to publish it, log in with `huggingface_hub` and call `trainer.push_to_hub()`.


In [ ]:
trainer.save_model("segformer-b0-suim-final")
image_processor.save_pretrained("segformer-b0-suim-final")


## References

- Hugging Face semantic segmentation tutorial: https://huggingface.co/docs/transformers/en/tasks/semantic_segmentation
- SUIM dataset: https://irvlab.cs.umn.edu/resources/suim-dataset
- SUIM paper: https://arxiv.org/abs/2004.01241
- SegFormer checkpoint: https://huggingface.co/nvidia/mit-b0
